# Combined Training: ISLES 2022 + SOOP

Train 3D U-Net on combined stroke dataset for improved Dice score.

**Datasets:**
- ISLES 2022: 250 subjects (DWI/ADC/FLAIR + lesion masks)
- SOOP (OpenNeuro ds004889): ~1,461 confirmed stroke subjects

**Baseline:** dice=0.606 (ISLES-only, no augmentation)

**Setup:**
1. Add dataset: `orvile/isles-2022-brain-stoke-dataset`
2. Add input: Your Work -> `03a_download_soop` notebook output (SOOP data)
3. Enable GPU: Settings -> Accelerator -> GPU T4 x2
4. Run all cells (Internet NOT required)

## 1. Setup

In [ ]:
# Clone repo
!git clone https://gitlab.com/Payz111/mri-stroke-assistance.git /kaggle/working/mri-stroke-assist
%cd /kaggle/working/mri-stroke-assist

In [ ]:
# Install dependencies
!pip install -q monai nibabel SimpleITK pyyaml

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Find datasets

ISLES = Kaggle dataset input, SOOP = notebook output input.

In [ ]:
import os
from pathlib import Path

# --- Find and unpack SOOP dataset (from 03a notebook output) ---
SOOP_ROOT = None

# First check if already unpacked
if os.path.exists("/tmp/soop/ds004889"):
    subs = [d for d in Path("/tmp/soop/ds004889").iterdir() if d.name.startswith("sub-")]
    if subs:
        SOOP_ROOT = Path("/tmp/soop/ds004889")
        print(f"SOOP already unpacked at {SOOP_ROOT}")

# If not unpacked, find and unpack the tar archive
if SOOP_ROOT is None:
    tar_path = None
    for item in Path("/kaggle/input").iterdir():
        # Check for tar file directly
        candidate = item / "soop_ds004889.tar"
        if candidate.exists():
            tar_path = candidate
            break
        # Check inside subdirs
        for f in item.rglob("soop_ds004889.tar"):
            tar_path = f
            break
        if tar_path:
            break

    if tar_path is None:
        print("Contents of /kaggle/input:")
        for item in Path("/kaggle/input").iterdir():
            print(f"  {item.name}/")
            if item.is_dir():
                for sub in list(item.iterdir())[:5]:
                    print(f"    {sub.name}")
        raise FileNotFoundError(
            "SOOP tar not found! Add the output of '03a_download_soop' notebook as Input."
        )

    print(f"Found SOOP archive: {tar_path}")
    print(f"Size: {tar_path.stat().st_size / 1e9:.1f} GB")
    print("Unpacking to /tmp/soop/ ...")
    os.makedirs("/tmp/soop", exist_ok=True)
    !tar xf {tar_path} -C /tmp/soop/
    SOOP_ROOT = Path("/tmp/soop/ds004889")
    print("Unpack complete!")

soop_subs = sorted([d.name for d in SOOP_ROOT.iterdir() if d.name.startswith("sub-")])
print(f"SOOP root: {SOOP_ROOT}")
print(f"SOOP subjects: {len(soop_subs)}")

# --- Find ISLES dataset ---
for candidate in [
    "/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset",
    "/kaggle/input/isles-2022-brain-stoke-dataset",
]:
    if os.path.exists(candidate):
        kaggle_input = Path(candidate)
        break
else:
    print("ISLES not found! Contents of /kaggle/input:")
    for item in os.listdir("/kaggle/input"):
        print(f"  {item}")
    raise FileNotFoundError("Add ISLES-2022 dataset in Kaggle settings.")

possible_roots = [kaggle_input / "ISLES-2022", kaggle_input]
isles_root = None
for root in possible_roots:
    if (root / "sub-strokecase0001").exists():
        isles_root = root
        break
if isles_root is None:
    for root, dirs, files in os.walk(kaggle_input):
        if "sub-strokecase0001" in dirs:
            isles_root = Path(root)
            break

isles_derivatives = isles_root / "derivatives"
print(f"\nISLES root: {isles_root}")
print(f"ISLES derivatives: {isles_derivatives}")

## 3. Test SOOP loading

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/mri-stroke-assist")

from src.data.soop_dataset import SOOPDataset
from src.data.transforms import get_val_transforms

# Discover subjects with complete data
print("Discovering SOOP subjects with complete data (DWI + ADC + FLAIR + mask)...")
soop_ds_all = SOOPDataset(data_root=SOOP_ROOT, require_mask=True)
print(f"SOOP subjects with complete data: {len(soop_ds_all)}")

# Test loading one sample
if len(soop_ds_all) > 0:
    soop_test = SOOPDataset(
        data_root=SOOP_ROOT,
        subject_ids=soop_ds_all.subject_ids[:1],
        transform=get_val_transforms(),
    )
    sample = soop_test[0]
    print(f"SOOP sample - image: {sample['image'].shape}, label: {sample['label'].shape}")
    print(f"Label sum: {sample['label'].sum():.0f} voxels")
    print("SOOP loading OK!")

## 4. Create combined dataset

In [ ]:
import json
from src.data.isles22_dataset import ISLES22Dataset
from src.data.soop_dataset import SOOPDataset
from src.data.combined_dataset import CombinedStrokeDataset
from src.data.transforms import get_train_transforms, get_val_transforms

# ISLES split
split_file = Path("/kaggle/working/mri-stroke-assist/data/splits/fold_0.json")
with open(split_file) as f:
    split = json.load(f)
print(f"ISLES fold 0: {split['n_train']} train, {split['n_val']} val")

# ISLES datasets
isles_train = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=split_file,
    split="train",
    transform=get_train_transforms(),
)
isles_val = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=split_file,
    split="val",
    transform=get_val_transforms(),
)

# SOOP training set (all subjects with masks)
soop_train = SOOPDataset(
    data_root=SOOP_ROOT,
    subject_ids=soop_ds_all.subject_ids,
    require_mask=True,
    transform=get_train_transforms(),
)

# Combined training set
train_combined = CombinedStrokeDataset([isles_train, soop_train])
print(f"\n{train_combined.summary()}")
print(f"Validation (ISLES only): {len(isles_val)}")
print(f"\nData increase: {len(isles_train)} -> {len(train_combined)} ({len(train_combined)/len(isles_train):.1f}x)")

## 5. Train!

In [ ]:
import logging
from torch.utils.data import DataLoader
from src.models.factory import create_model, create_loss
from src.train.trainer import Trainer
from src.train.callbacks import CheckpointCallback, EarlyStoppingCallback

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s", datefmt="%H:%M:%S")

# Training config
EPOCHS = 100
BATCH_SIZE = 4
LR = 1e-4
PATIENCE = 20
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Config: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, patience={PATIENCE}")
print(f"Train: {len(train_combined)}, Val: {len(isles_val)}")
print(f"Device: {DEVICE}")

In [ ]:
# DataLoaders
train_loader = DataLoader(
    train_combined, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    isles_val, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Model (same architecture as baseline)
model_cfg = {"name": "unet3d", "in_channels": 3, "out_channels": 1, "features": [32, 64, 128, 256], "dropout": 0.1}
loss_cfg = {"type": "dice_focal", "dice_weight": 0.5, "focal_weight": 0.5, "focal_gamma": 2.0}

model = create_model(model_cfg)
criterion = create_loss(loss_cfg)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer & scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# Callbacks
output_dir = Path("/kaggle/working/outputs/combined")
output_dir.mkdir(parents=True, exist_ok=True)

callbacks = [
    CheckpointCallback(save_dir=output_dir / "checkpoints", monitor="val_dice"),
    EarlyStoppingCallback(patience=PATIENCE, monitor="val_dice"),
]

In [ ]:
import time

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    scheduler=scheduler,
    callbacks=callbacks,
)

t0 = time.time()
result = trainer.fit(num_epochs=EPOCHS)
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} min")
print(f"Best val_dice: {result['best_val_dice']:.4f} at epoch {result['best_epoch'] + 1}")
print(f"\n--- Comparison ---")
print(f"Baseline (ISLES only, 200 train): dice=0.6062")
print(f"Combined (ISLES+SOOP):            dice={result['best_val_dice']:.4f}")
print(f"Change: {result['best_val_dice'] - 0.6062:+.4f}")

## 6. Training curves

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs_range = [h["epoch"] + 1 for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, [h["train_loss"] for h in history], label="Train")
axes[0].plot(epochs_range, [h["val_loss"] for h in history], label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_range, [h["train_dice"] for h in history], label="Train")
axes[1].plot(epochs_range, [h["val_dice"] for h in history], label="Val")
axes[1].axhline(y=0.606, color="r", linestyle="--", alpha=0.5, label="Baseline (0.606)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].set_title("Dice Score")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("/kaggle/working/outputs/combined_training_curves.png", dpi=150)
plt.show()
print(f"Best val Dice: {result['best_val_dice']:.4f}")

## 7. Save results

In [ ]:
import json

history_path = output_dir / "training_history.json"
with open(history_path, "w") as f:
    json.dump(result["history"], f, indent=2)

print(f"Checkpoint: {output_dir / 'checkpoints' / 'best_model.pth'}")
print(f"History: {history_path}")
print(f"Curves: /kaggle/working/outputs/combined_training_curves.png")
print("\nDownload these files from the Output tab!")